# Clase 032 — eval y query

**Parte 0** · VanderPlas cap. 3 § 3.13.

> 🎯 Filtros y cálculos como strings tipo SQL. Útil para legibilidad y datasets grandes.

> ⏱️ ~45 min

## ⚙️ Setup

In [ ]:
import numpy as np
import pandas as pd
import time
rng = np.random.default_rng(42)

## 1️⃣ `df.query` — filtro como string

In [ ]:
df = pd.DataFrame({
    'precio'    : rng.uniform(10, 1000, 20).round(2),
    'cantidad'  : rng.integers(1, 50, 20),
    'categoria' : rng.choice(['A', 'B', 'C'], 20),
})

# Filtro tradicional
mask = (df['precio'] > 100) & (df['cantidad'] < 30) & (df['categoria'] == 'A')
filtrado_a = df[mask]

# Equivalente con query
filtrado_b = df.query('precio > 100 and cantidad < 30 and categoria == "A"')

print('mismo resultado:', filtrado_a.equals(filtrado_b))
print(filtrado_b)

## 2️⃣ Variables locales con `@`

Referencia variables del scope con prefijo `@`:

In [ ]:
threshold_precio = 500
categoria_objetivo = 'A'

result = df.query('precio > @threshold_precio and categoria == @categoria_objetivo')
print(result)

## 3️⃣ `df.eval` — expresiones aritméticas

Calcula columnas sin temporales y, en datasets grandes, usando `numexpr` (más rápido).

In [ ]:
# Tradicional
df['total_a'] = df['precio'] * df['cantidad']

# Con eval
df['total_b'] = df.eval('precio * cantidad')

# inplace: añade al DataFrame
df.eval('descuento = precio * 0.1', inplace=True)

print(df[['precio', 'cantidad', 'total_a', 'total_b', 'descuento']].head())
print('\niguales total_a == total_b?', (df['total_a'] == df['total_b']).all())

## 4️⃣ Cuándo conviene query/eval

**Sí**:
- Cadenas de filtros largas → más legible una string que `(a) & (b) & (c) & (d)`.
- Datasets grandes (>10k filas) con expresiones complejas → `numexpr` da speedup.
- Filtros parametrizables (con `@`) sin construir máscaras complejas.

**No**:
- Datasets pequeños — overhead del parser no compensa.
- Cuando necesitas autocomplete del IDE — strings no se autocompletan.
- Cuando el filtro usa métodos custom (no es solo aritmética/comparación).

## 5️⃣ Benchmark — speedup en grandes

In [ ]:
N = 1_000_000
big = pd.DataFrame({
    'a': rng.normal(0, 1, N),
    'b': rng.normal(0, 1, N),
    'c': rng.choice(['x','y','z'], N),
})

t0 = time.perf_counter()
_ = big[(big['a'] > 0.5) & (big['b'] < -0.5) & (big['c'] == 'x')]
t1 = time.perf_counter()

t2 = time.perf_counter()
_ = big.query('a > 0.5 and b < -0.5 and c == "x"')
t3 = time.perf_counter()

print(f'tradicional : {(t1-t0)*1000:.1f} ms')
print(f'query       : {(t3-t2)*1000:.1f} ms')

## ✅ Checklist

- [ ] Sé escribir filtros largos con `df.query`
- [ ] Uso `@var` para referenciar variables locales
- [ ] Uso `df.eval` para columnas derivadas sin temporales
- [ ] Sé que el speedup aparece en datasets grandes
- [ ] Reconozco trade-off: legibilidad vs autocomplete IDE

## 📝 Homework

Ver `README.md`. 3 filtros equivalentes, eval para cols, benchmark en 1M filas.

## 🔗 Referencias

- VanderPlas cap. 3 § 3.13
- [pandas enhancing perf](https://pandas.pydata.org/docs/user_guide/enhancingperf.html)

➡️ **Siguiente:** [033 — Matplotlib: anatomía figura/axes](../033-matplotlib-anatomia-figura-axes/README.md)